In [21]:
import socket
import struct
import time
from pathlib import Path
import base64

from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.asymmetric.utils import decode_dss_signature
from cryptography.hazmat.primitives.asymmetric import utils
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

In [22]:
def generateServerKeys():
    file_path_secret=Path("server_private_key.der")
    file_path_public=Path("server_public_key.der")

    if not file_path_public.is_file() and not file_path_secret.is_file():
        print("generating...")
        # Generate server identity key (EC)
        server_identity_key = ec.generate_private_key(ec.SECP256R1())
        server_identity_public = server_identity_key.public_key()

        secret_bytes=server_identity_key.private_bytes(
                encoding=serialization.Encoding.DER,
                format=serialization.PrivateFormat.PKCS8,
                encryption_algorithm=serialization.NoEncryption()  # or best: use a passphrase
            )
        pub_bytes=server_identity_public.public_bytes(
                encoding=serialization.Encoding.DER,
                format=serialization.PublicFormat.SubjectPublicKeyInfo
            )

        # Save private key to file (DER format)
        with open("server_private_key.txt", "wb") as f:
            f.write(base64.b64encode(secret_bytes).decode("ascii"))

        # Save public key to file (DER format)
        with open("server_public_key.txt", "wb") as f:
            f.write(base64.b64encode(pub_bytes).decode("ascii"))

generateServerKeys()

In [ ]:
def load_server_private():
# Load private key
    with open("server_private_key.txt", "rb") as f:
        server_identity_key = serialization.load_der_private_key(f.read(), password=None)
    return server_identity_key

def load_server_public():
    # Load public key
    with open("server_public_key.txt", "rb") as f:
        server_identity_public = serialization.load_der_public_key(f.read())
    return server_identity_public

print(load_server_public())
print(load_server_private())